# PV + Heat Pump System Optimization (MILP)

**Author:** Agus Samsudin  
**Domain:** Energy Systems Modelling &nbsp;·&nbsp; Optimization &nbsp;·&nbsp; Renewable Energy  
**Tools:** Python · Pyomo · GLPK · Pandas · NumPy · Matplotlib · Plotly

---

## Overview

This notebook implements a **Mixed Integer Linear Programming (MILP)** model
to optimize the sizing and operation of a residential or commercial building
hybrid energy system.

The model determines optimal **investment decisions** (technology capacities)
and **operational dispatch** of electricity supply technologies to meet hourly
building demand at **minimum total system cost**, achieving the lowest possible
**Levelized Cost of Electricity (LCOE)**.

---

## System Components

| Component | Role | Optional |
|---|---|---|
| **Solar PV** | On-site renewable electricity generation | No |
| **Battery Storage** | Short-term energy buffer; increases PV self-consumption | Yes |
| **Electric Heat Pump** | Replaces gas/oil boiler; converts electricity to heat | Yes |
| **Grid Connection** | Backup electricity supply when local generation is insufficient | Yes |

---

## Objective

Minimise the **total annualised system cost**, which includes:

- Capital investment cost (PV, battery, heat pump)
- Fixed operation and maintenance (O&M)
- Variable grid electricity cost
- Battery cycling degradation cost

This is equivalent to minimising the **Levelized Cost of Electricity (LCOE)**
of the hybrid system.

---

## Model Workflow

| Step | Description |
|:---:|---|
| **1** | Import required Python libraries |
| **2** | Load geographic project information |
| **3** | Configure system technologies and economic parameters |
| **4** | Load hourly time series data (demand, PV, heat pump) |
| **5** | Build and solve the MILP optimization model |
| **6** | Analyse and visualize results |

---

## Installation

```bash
pip install -r requirements.txt
```

**GLPK Solver:**
```bash
# Ubuntu / Debian
sudo apt install glpk-utils

# macOS
brew install glpk

# Windows — download from https://winglpk.sourceforge.net/
```

---

> **Disclaimer:** This model was developed for educational and research purposes.
> While every effort has been made to ensure accuracy, the author makes no guarantees
> regarding completeness or reliability of results. Use at your own risk.

---

## Step 1 — Required Libraries

The following Python packages are required to run this notebook:

| Library | Purpose |
|---|---|
| **Pandas** | Data loading, handling, and time-series processing |
| **NumPy** | Numerical operations and array handling |
| **Matplotlib** | Visualization of dispatch, capacity, and economic results |
| **Plotly** | Interactive Sankey energy flow diagram |
| **Pyomo** | Optimization modelling framework for the MILP formulation |
| **ipywidgets** | Interactive configuration interface inside the notebook |

The model uses the **GLPK (GNU Linear Programming Kit)** solver to solve
the MILP optimization problem. GLPK is open-source and free to use.

A global matplotlib style is applied here so that all charts in this
notebook share a consistent, professional appearance.

### Project Structure

All model logic lives in the `src/` folder as separate Python modules:

| Module | Description |
|---|---|
| `src/constants.py` | All techno-economic assumptions (single source of truth) |
| `src/geographic.py` | Project site metadata loader |
| `src/setup.py` | Interactive system configuration (ipywidgets) |
| `src/timeseries.py` | Hourly profile loading and validation |
| `src/model.py` | Pyomo MILP formulation (build + solve) |
| `src/visualization.py` | Results summaries and publication-quality charts |

In [ ]:
# ── Core libraries ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

# ── Import model modules from src/ folder ─────────────────────────────────────
from src.constants     import *
from src.geographic    import GeographicDescription
from src.setup         import SetupOptions
from src.timeseries    import TimeSeriesData
from src.model         import BuildingEnergyMILP
from src.visualization import ResultsVisualization

# ── Global matplotlib style ────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'font.size':          11,
    'axes.titlesize':     13,
    'axes.titleweight':   'bold',
    'axes.labelsize':     11,
    'legend.fontsize':    10,
    'xtick.labelsize':    10,
    'ytick.labelsize':    10,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.grid':          True,
    'grid.color':         '#e5e5e5',
    'grid.linewidth':     0.7,
    'axes.facecolor':     '#fafafa',
    'figure.facecolor':   'white',
    'lines.linewidth':    1.8,
})

print("All modules loaded successfully.")

---

## Model Constants

All techno-economic assumptions are defined in `src/constants.py`.
This is the **only place** where assumptions need to be changed —
no values are hard-coded or duplicated elsewhere in the project.

### Battery Parameters

| Constant | Value | Source / Notes |
|---|---|---|
| `BATTERY_EFFICIENCY` | 0.90 | Round-trip charge/discharge efficiency (typical Li-ion) |
| `SOC_MIN_FRACTION` | 0.20 | Minimum state-of-charge — prevents deep discharge damage |
| `BATTERY_C_RATE` | 3 | C-rate denominator: max charge/discharge power = capacity / 3 |
| `BATTERY_CYCLE_LIFE` | 4000 | Cycle life at 80 % DoD (typical Li-ion NMC/LFP) |

### Technology Lifetimes

| Constant | Value | Source |
|---|---|---|
| `PV_LIFETIME_YRS` | 25 | IEA standard assumption for utility/rooftop PV |
| `BATTERY_LIFETIME_YRS` | 15 | Typical Li-ion warranty and economic lifetime |

### Heat Pump Parameters

| Constant | Value | Notes |
|---|---|---|
| `HP_COP` | 3.0 | Coefficient of Performance — air-source heat pump, moderate climate |
| `HP_KW_PER_M2` | 0.06 | Sizing rule: 60 W per m² floor area (standard European assumption) |

### O&M

| Constant | Value | Notes |
|---|---|---|
| `PV_OPEX_FRACTION` | 0.02 | Annual O&M cost as a fraction of CAPEX (2 %) |

---

## Step 2 — Geographic Information

This section loads geographic information about the location of the
building or project under analysis.

The geographic data is not directly used in the optimization model, but
provides useful **contextual documentation** for project reporting.
In future versions, the coordinates could be used to automatically
retrieve solar irradiance and weather data via APIs such as
[PVGIS](https://re.jrc.ec.europa.eu/pvg_tools/) or
[Renewables.ninja](https://www.renewables.ninja).

**Expected file:** `data/geographic.csv`

**Required columns:**

| Column | Type | Description |
|---|---|---|
| `Name` | string | Location name (e.g. city or project name) |
| `Latitude` | float | Decimal degrees north |
| `Longitude` | float | Decimal degrees east |

**Example:**

```
Name,Latitude,Longitude
Berlin,52.52,13.41
```

In [ ]:
geo = GeographicDescription()
geo.upload()

---

## Step 3 — System Configuration

This section allows the user to define the configuration of the building
energy system before running the optimization.

### Technology Selection

Users can activate or deactivate the following technologies:

| Technology | Description |
|---|---|
| **Solar PV** | On-site photovoltaic generation; capacity is optimized by the model |
| **Battery Storage** | Electrochemical energy storage; shifts PV generation to periods of demand |
| **Heat Pump** | Electric heat pump replacing conventional gas or oil boiler |
| **Grid Backup** | Grid electricity import; if disabled, the system operates in off-grid mode |

### Economic Parameters

The following economic inputs determine the cost structure of the optimization:

| Parameter | Unit | Description |
|---|---|---|
| **Currency** | — | Currency symbol used in all outputs (e.g. EUR, USD) |
| **Electricity price** | currency/kWh | Grid import tariff |
| **PV CAPEX** | currency/kW | Installed cost per kW of PV capacity |
| **Battery CAPEX** | currency/kWh | Installed cost per kWh of battery capacity |
| **Heat Pump CAPEX** | currency/kW | Installed cost per kW of heat pump thermal capacity |
| **Discount rate** | — | Weighted average cost of capital (WACC); used in CRF calculation |
| **Annual heating cost** | currency/yr | Current annual gas or oil heating bill (used for heat pump comparison) |

### Building Parameters

| Parameter | Unit | Description |
|---|---|---|
| **Living space** | m² | Floor area used to size the heat pump (60 W/m² rule) |

Click **Confirm Setup** when all parameters have been entered.
The currency defined here will be used consistently across all results outputs.

In [ ]:
setup = SetupOptions()
setup.display()

---

## Step 4 — Time Series Data

This section loads the hourly time series data required by the optimization model.
All datasets must contain exactly **8760 hourly values** representing one full year.

### Required Datasets

| File | Unit | Description |
|------|------|-------------|
| `data/pv_generation.csv` | kW / kWp | Normalised PV output per kWp installed (0–1 range) |
| `data/electricity_demand.csv` | kW | Hourly building electricity demand (not scaled) |
| `data/heatpump_load.csv` | normalised 0–1 | Hourly heating load profile, scaled by heat pump size |

### Data Sources

Hourly generation profiles and weather-driven load data can be obtained from:

- **[Renewables.ninja](https://www.renewables.ninja)** — PV generation profiles and wind data
- **[Global Solar Atlas](https://globalsolaratlas.info)** — solar irradiance data
- **[PVGIS](https://re.jrc.ec.europa.eu/pvg_tools/)** — European Commission PV estimation tool
- **[DWD / Meteonorm](https://meteonorm.com)** — measured meteorological time series

### Heat Pump Electricity Demand

Heat pump electricity demand is computed **once** in this step as a `total_demand`
property and reused by both the MILP model (Step 5) and the results visualizer (Step 6).
This eliminates duplicated logic and ensures both steps always use identical values.

The heat pump electricity demand is derived as:

$$P_{\text{HP,elec}}(t) = \frac{P_{\text{heat}}(t) \times A_{\text{floor}} \times k_{\text{W/m}^2}}{\text{COP}}$$

where $k_{\text{W/m}^2} = 0.06$ kW/m² and COP = 3.0 (default, configurable in constants).

In [ ]:
print("Loading time series data...")
ts = TimeSeriesData(setup)
ts.upload_generation()
ts.upload_demand()
ts.upload_heatpump()
print("Done.")

---

## Step 5 — Optimization Model (MILP)

The system is optimized using a **Mixed Integer Linear Programming (MILP)**
formulation implemented in [Pyomo](http://www.pyomo.org/) and solved with the
open-source **GLPK** solver.

### Decision Variables

| Variable | Unit | Description |
|---|---|---|
| `PVCap` | kW | Installed PV capacity (investment decision) |
| `BatteryEnergyCap` | kWh | Installed battery energy capacity (investment decision) |
| `GridImport[t]` | kW | Grid electricity import at each hour |
| `Charge[t]` | kW | Battery charging power at each hour |
| `Discharge[t]` | kW | Battery discharging power at each hour |
| `SOC[t]` | kWh | Battery state of charge at each hour |
| `PVCurtail[t]` | kW | Curtailed PV generation at each hour |

### Objective Function

Minimise total annualised system cost:

$$\min \; \underbrace{\sum_t \text{GridImport}_t \cdot p_{\text{grid}}}_{\text{grid cost}}
+ \underbrace{\sum_t \text{Discharge}_t \cdot c_{\text{cycle}}}_{\text{battery cycling}}
+ \underbrace{\text{PVCap} \cdot (\text{CRF}_{PV} \cdot K_{PV} + O\&M_{PV})}_{\text{PV annualised CAPEX}}
+ \underbrace{\text{BattCap} \cdot \text{CRF}_{batt} \cdot K_{batt}}_{\text{battery annualised CAPEX}}$$

where the **Capital Recovery Factor (CRF)** annualises investment cost over
the technology lifetime at the given discount rate:

$$\text{CRF} = \frac{r(1+r)^n}{(1+r)^n - 1}$$

### Key Constraints

| Constraint | Description |
|---|---|
| **Power balance** | PV + Grid + Discharge = Demand + Charge + Curtailment (every hour) |
| **SoC dynamics** | $\text{SoC}_t = \text{SoC}_{t-1} + \eta \cdot \text{Charge}_t - \text{Discharge}_t / \eta$ |
| **Cyclic SoC** | End-of-year state of charge equals start-of-year (no stored energy carry-over) |
| **SoC limits** | $0.2 \cdot E_{\text{batt}} \leq \text{SoC}_t \leq E_{\text{batt}}$ |
| **C-rate limits** | Charge and discharge power limited to capacity / 3 |
| **Grid off-grid mode** | Grid import forced to zero if Grid Backup is deactivated |

### Storage Round-Trip Efficiency

Battery charging and discharging losses are modelled using a round-trip
efficiency $\eta = 0.90$, applied symmetrically to both charge and discharge:

$$\text{SoC}_t = \text{SoC}_{t-1} + \eta \cdot P_{\text{charge},t} - \frac{P_{\text{discharge},t}}{\eta}$$

In [ ]:
milp = BuildingEnergyMILP(setup, ts)
milp.build()
milp.solve()

---

## Step 6 — Results Visualization

After solving the optimization problem, results are summarized and visualized
through text summaries.

### Text Summaries

| Method | Description |
|--------|-------------|
| `summary()` | Optimal installed capacity of each technology |
| `capex()` | Total capital investment breakdown by technology |
| `annual_energy()` | Annual energy balance including self-sufficiency & self-consumption |
| `economics()` | Cost comparison, payback period, LCOE, and LCOS |

### Charts

| # | Chart | Description |
|---|-------|-------------|
| 1 | **KPI Dashboard** | Gauge-style overview: self-sufficiency, self-consumption, LCOE, payback |
| 2 | **Energy Supply Mix** | Donut chart with self-sufficiency centre KPI |
| 3 | **Battery Charging** | Donut chart: PV vs grid charging sources |
| 4 | **Monthly Balance** | Stacked bar: monthly supply vs demand comparison |
| 5 | **Hourly Heatmap** | 24h × 365d surplus/deficit heatmap |
| 6 | **Cost Comparison** | Stacked bar: baseline vs optimised annual cost |
| 7 | **CAPEX Waterfall** | Investment breakdown waterfall chart |
| 8 | **Hourly Dispatch** | Stacked area: PV routing, battery, grid, curtailment |
| 9 | **Battery SoC** | State of charge with min/max capacity bands |
| 10 | **Energy Sankey** | Full flow: source → storage → load → losses |

Run the cell below to generate **all results at once**.
Figures are saved to `result/` automatically.

In [ ]:
# ── Run all results ────────────────────────────────────────────────────────────
viz = ResultsVisualization(milp, setup, ts)
viz.plot_all(hours=(4032, 4200), save_dir="result")

---

## Model Summary

This notebook demonstrates a **techno-economic optimization** of a hybrid
building energy system using Mixed Integer Linear Programming (MILP).

The model determines the least-cost configuration of:

- **Solar PV capacity** — optimal installed kW based on generation profile and demand
- **Battery storage capacity** — optimal kWh sized to maximize PV self-consumption
- **Grid electricity imports** — minimized through on-site generation and storage
- **Heat pump operation** — optional electrification of building heating

while satisfying hourly electricity demand and all technical constraints.

---

### Potential Extensions

The model structure is designed to be **modular and adaptable**.
Future versions could incorporate:

| Extension | Description |
|---|---|
| Electric vehicle charging | Add EV as a flexible load or V2G storage asset |
| Dynamic electricity tariffs | Time-of-use or real-time pricing signals |
| Demand response | Shiftable loads to improve system flexibility |
| CO₂ minimization | Dual objective or emission constraint |
| Multi-building aggregation | District-level or virtual power plant modelling |
| Stochastic optimization | Uncertainty in demand, solar, and prices |

---

### References

- Lofberg, J. (2004). YALMIP: A Toolbox for Modeling and Optimization in MATLAB.
- Hart, W.E. et al. (2017). Pyomo – Optimization Modeling in Python. Springer.
- IRENA (2023). Renewable Power Generation Costs. International Renewable Energy Agency.
- IEA (2023). World Energy Outlook. International Energy Agency.

---

*Developed by Agus Samsudin — Energy Systems Modelling*